<b>Preparation</b>

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Question 1

In [3]:
len(documents)

72

In [4]:
documents[10] #random

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

Question 2

In [5]:
from minsearch import Index
index=Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [6]:
index.search("How does the agentic loop keep calling the model until it stops?",num_results=1)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

Question 3

In [7]:
from rag_helper2 import RAGbase

In [8]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client=OpenAI()

In [9]:
assistant=RAGbase(index,openai_client)

In [10]:
question= assistant.rag("How does the agentic loop keep calling the model until it stops?")
question.output_text

'It keeps calling the model in a `while True` loop, and after each response it checks whether the response contained any `function_call` items.\n\n- If there are function calls, it runs the tools, appends the tool outputs to `messages`, and loops again.\n- If there are no function calls, it `break`s out of the loop.\n\nSo the stop condition is: **no function calls in the model’s response**.'

In [11]:
question.usage.input_tokens

7126

Question 4

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

Question 5

In [14]:
from minsearch import Index
ch_index=Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

ch_index.fit(chunks)

In [15]:
assistant2=RAGbase(ch_index,openai_client)

In [16]:
q5=assistant2.rag("How does the agentic loop keep calling the model until it stops?")
q5.usage.input_tokens

2309

Question 6